# 03 — Метрики оценки результатов кластеризации

# Кластеризация данных: практическая серия ноутбуков

сначала формулируется идея метода, затем показывается его геометрический смысл, визуализация и практическая реализация в Python. В исходном уроке акцент сделан на объяснении метода через визуальные представления и двумерные проекции; здесь тот же подход перенесён на алгоритмы кластеризации.

## 1. Зачем оценивать кластеризацию?

В обучении без учителя нет одного очевидного `accuracy`. Нужно ответить на два разных вопроса:

1. **Внутренняя оценка:** хорошо ли сами точки разделены в пространстве признаков?
2. **Внешняя оценка:** насколько найденные кластеры совпадают с уже известными истинными метками, если такие метки существуют?

`Silhouette score` относится к первой группе. `Homogeneity`, `Completeness`, `V-measure`, `ARI` и `NMI` используют истинные метки классов и относятся к внешней оценке.

## 2. Silhouette score

Для объекта $i$:
- $a(i)$ — среднее расстояние до точек своего кластера;
- $b(i)$ — минимальное среднее расстояние до точек другого кластера.

$$
s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}
$$

Итоговый silhouette — среднее значение $s(i)$.

**Диапазон:** от `-1` до `1`.

- близко к `1`: точка хорошо отделена;
- около `0`: граница между кластерами;
- меньше `0`: точка может быть отнесена не к тому кластеру.

**Преимущества:** не нужны истинные labels; удобно сравнивать разные `K`.

**Недостатки:** зависит от расстояния и геометрии; может предпочитать компактные выпуклые кластеры; шум и неоднородная плотность осложняют интерпретацию.

**Условия:** нужны `X` и найденные `cluster_labels`; минимум два кластера.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    homogeneity_score,
    completeness_score,
    v_measure_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.preprocessing import StandardScaler

X, y_true = make_blobs(
    n_samples=600, centers=3, cluster_std=1.1, random_state=42
)
X = StandardScaler().fit_transform(X)

pred = KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(X)

print("Silhouette:", silhouette_score(X, pred))

## 3. Homogeneity

Кластеризация **гомогенна**, если каждый найденный кластер содержит объекты преимущественно одного истинного класса.

Формально показатель строится через условную энтропию:

$$
h = 1 - \frac{H(C|K)}{H(C)}
$$

где `C` — истинный класс, `K` — найденный кластер.

**Диапазон:** `[0, 1]`; `1` означает идеальную гомогенность.

**Плюс:** показывает, насколько кластер «чистый».

**Минус:** высокая homogeneity не гарантирует, что один истинный класс не был раздроблен на множество кластеров.

**Условия:** необходимы `y_true` и `y_pred`; метки должны описывать одни и те же объекты.

## 4. Completeness

**Completeness** проверяет обратную сторону: все ли объекты одного истинного класса попали в один и тот же найденный кластер.

$$
c = 1 - \frac{H(K|C)}{H(K)}
$$

**Диапазон:** `[0, 1]`.

**Преимущество:** обнаруживает раздробление одного истинного класса.

**Недостаток:** не штрафует так же сильно за смешение разных классов внутри кластера, как homogeneity.

**Условия:** нужны истинные метки и предсказанные кластерные метки.

## 5. V-measure

**V-measure** объединяет homogeneity и completeness через гармоническое среднее:

$$
V = 2\frac{hc}{h+c}
$$

Если одна из компонент низкая, итоговый показатель также снижается.

**Преимущества:** одно число для оценки одновременно чистоты и полноты; симметричная оценка.

**Недостатки:** без истинных labels применять нельзя; одно число может скрывать, какая именно сторона проблемы — homogeneity или completeness.

**Условия:** нужны `y_true` и `y_pred`.

## 6. Adjusted Rand Index (ARI)

Rand Index оценивает, насколько согласованы решения о том, какие пары объектов находятся в одной/разных группах. **ARI** дополнительно корректирует результат на согласование, ожидаемое случайно.

**Интерпретация:**
- `1` — идеальное совпадение;
- `0` — примерно уровень случайного совпадения;
- значения ниже `0` возможны, если согласование хуже ожидаемого случайного.

**Преимущества:** не зависит от конкретных числовых названий кластеров; хорошо работает для сравнения разбиений.

**Недостатки:** требует истинной разметки; при очень несбалансированных классах интерпретация может быть сложнее.

**Условия:** нужны `y_true` и `y_pred`, причём длины должны совпадать.

## 7. Normalized Mutual Information (NMI)

NMI измеряет взаимную информацию между истинным и найденным разбиением и нормализует её:

$$
NMI = \frac{I(C;K)}
{\sqrt{H(C)H(K)}}
$$

В стандартной реализации `scikit-learn` показатель лежит от `0` до `1`.

- `0` — разбиения практически независимы;
- `1` — полное совпадение информации о разбиении.

**Преимущества:** не зависит от перестановки номеров кластеров; измеряет информационное соответствие.

**Недостатки:** требует истинных labels; значение зависит от способа нормализации и может быть менее интуитивно, чем ARI.

**Условия:** нужны `y_true` и `y_pred`.

In [ ]:
metrics = {
    "Homogeneity": homogeneity_score(y_true, pred),
    "Completeness": completeness_score(y_true, pred),
    "V-measure": v_measure_score(y_true, pred),
    "ARI": adjusted_rand_score(y_true, pred),
    "NMI": normalized_mutual_info_score(y_true, pred),
}
display(pd.Series(metrics).to_frame("score"))

## 8. Сравнение метрик

| Метрика | Нужны истинные labels? | Что измеряет |
|---|---:|---|
| Silhouette | Нет | компактность + отделённость в пространстве |
| Homogeneity | Да | чистоту найденных кластеров |
| Completeness | Да | целостность истинных классов |
| V-measure | Да | баланс homogeneity/completeness |
| ARI | Да | согласованность пар объектов с поправкой на случайность |
| NMI | Да | взаимную информацию между двумя разбиениями |

### Важный практический вывод

Если `y_true` нет, нельзя честно посчитать ARI/NMI/Homogeneity/Completeness/V-measure. Для реальной кластеризации клиентов банка это особенно важно: `exit`, `customer_segment`, `risk_segment` могут использоваться **только как внешние контрольные признаки**, а не как «истина», если они были сформированы из тех же исходных признаков.

Кроме того, номера кластеров произвольны: `cluster 0` и `cluster 1` можно поменять местами без изменения качества. ARI/NMI и родственные метрики это учитывают.

## 9. Псевдоалгоритм silhouette

```python
для каждой точки i:
    a = среднее расстояние от i до своего кластера
    для каждого другого кластера:
        вычислить среднее расстояние до него
    b = минимальное из этих средних расстояний
    silhouette_i = (b - a) / max(a, b)

итог = среднее silhouette_i
```

Псевдокод показывает, почему метрика одновременно учитывает **внутрикластерную компактность** и **межкластерное разделение**.